In [0]:
# Inicialização de algumas constantes

catalog = "cinedata_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"


bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/landing"

# Nome das tabelas da camada
tb_infos = "tb_movies_info"
tb_financials = "tb_movies_financials"
tb_metrics = "tb_movies_metrics"
tb_credits_and_tags = "tb_credits_and_tags"
tb_reviews = "tb_movies_reviews"

# Mapeamento dos caminhos para os arquivos brutos
path_infos = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_credits_and_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_reviews = f"{landing_path}/movies_reviews.csv"


In [0]:

from pyspark.sql import functions as F

def find_col(df, candidates, required=False):
    lookup = {c.lower().strip(): c for c in df.columns}

    for name in candidates:
        if name.lower().strip() in lookup:
            return lookup[name.lower().strip()]

    if required:
        raise ValueError(
            f"Coluna não encontrada: {candidates}. "
            f"Colunas disponíveis: {df.columns}"
        )

    return None


def value_col(df, candidates, dtype, required=False):
    name = find_col(df, candidates, required)

    if name is None:
        return F.lit(None).cast(dtype)

    return F.col(f"`{name}`").cast(dtype)


def date_col(df, candidates):
    name = find_col(df, candidates)

    if name is None:
        return F.lit(None).cast("date")

    raw = F.trim(F.col(f"`{name}`").cast("string"))

    # Aceita MM-dd-yyyy e yyyy-MM-dd.
    # Formatos inválidos retornam NULL.
    return F.coalesce(
        F.to_date(
            F.try_to_timestamp(raw, F.lit("MM-dd-yyyy"))
        ),
        F.to_date(
            F.try_to_timestamp(raw, F.lit("yyyy-MM-dd"))
        ),
        F.to_date(
            F.try_to_timestamp(raw, F.lit("yyyy-MM-dd HH:mm:ss"))
        )
    )


def surrogate_key(df, natural_keys, sk_name):
    # Gera uma SK BIGINT determinística a partir das chaves naturais.
    # Utiliza 60 bits do hash; valide a unicidade após a criação.
    key_json = F.to_json(
        F.struct(*[F.col(c) for c in natural_keys])
    )

    return df.withColumn(
        sk_name,
        F.conv(
            F.substring(F.sha2(key_json, 256), 1, 15),
            16,
            10
        ).cast("long")
    )


def overwrite_table(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{gold_schema}.{table_name}")
    )


In [0]:

from pyspark.sql import functions as F

movies_src = spark.table(f"{bronze_schema}.tb_movies_info")


# Metadados dos filmes

movies = movies_src.select(
    value_col(
        movies_src,
        ["id_filme", "movie_id", "tmdb_id", "id"],
        "string",
        True
    ).alias("id_filme"),

    value_col(
        movies_src,
        ["titulo", "title", "original_title", "movie_title"],
        "string"
    ).alias("titulo"),

    date_col(
        movies_src,
        ["data_lancamento", "release_date", "released"]
    ).alias("data_lancamento"),

    F.expr(
        f"try_cast(`{find_col(movies_src, ['duracao_minutos', 'runtime', 'duration', 'runtime_minutes'])}` AS INT)"
        if find_col(movies_src, ['duracao_minutos', 'runtime', 'duration', 'runtime_minutes'])
        else "CAST(NULL AS INT)"
    ).alias("duracao_minutos"),

    value_col(
        movies_src,
        ["idioma_original", "original_language", "language"],
        "string"
    ).alias("idioma_original"),

    value_col(
        movies_src,
        ["status_filme", "status"],
        "string"
    ).alias("status_filme"),

    value_col(
        movies_src,
        ["sinopse", "overview", "plot", "description"],
        "string"
    ).alias("sinopse")
).filter(
    F.col("id_filme").isNotNull()
).dropDuplicates(["id_filme"])

movies = movies.withColumn(
    "ano_lancamento",
    F.year("data_lancamento").cast("int")
)

dim_movies = surrogate_key(
    movies,
    ["id_filme"],
    "sk_movie_id"
).select(
    "sk_movie_id",
    "id_filme",
    "titulo",
    "data_lancamento",
    "ano_lancamento",
    "duracao_minutos",
    "idioma_original",
    "status_filme",
    "sinopse"
)

overwrite_table(dim_movies, "dim_movies")

display(dim_movies.limit(10))


In [0]:

genres_src = spark.table(f"{silver_schema}.tb_generos")

dim_genres_base = genres_src.select(
    F.trim(
        value_col(
            genres_src, ["nome_genero", "genero", "genre", "genres"],
            "string", True
        )
    ).alias("nome_genero")
).filter(
    F.col("nome_genero").isNotNull() &
    (F.col("nome_genero") != "")
).dropDuplicates(["nome_genero"])

dim_genres = surrogate_key(
    dim_genres_base,
    ["nome_genero"],
    "sk_genre_id"
).select(
    "sk_genre_id",
    "nome_genero"
)

overwrite_table(dim_genres, "dim_genres")

display(dim_genres)


In [0]:

entities_src = spark.table(f"{silver_schema}.tb_pessoas_empresas")

entities = entities_src.select(
    value_col(
        entities_src, ["id_filme", "movie_id"],
        "string", True
    ).alias("id_filme"),

    F.trim(
        value_col(
            entities_src, ["nome_entidade", "nome_pessoa", "name"],
            "string", True
        )
    ).alias("nome_entidade"),

    F.initcap(
        F.trim(
            value_col(
                entities_src, ["tipo_entidade", "tipo_pessoa", "type"],
                "string", True
            )
        )
    ).alias("tipo_entidade")
).filter(
    F.col("id_filme").isNotNull() &
    F.col("nome_entidade").isNotNull() &
    (F.col("nome_entidade") != "") &
    F.col("tipo_entidade").isNotNull()
)

person_types = ["Ator", "Diretor", "Roteirista"]

display(entities.limit(10))


In [0]:

people = entities.filter(
    F.col("tipo_entidade").isin(person_types)
).select(
    F.col("nome_entidade").alias("nome_pessoa"),
    F.col("tipo_entidade").alias("tipo_pessoa")
).dropDuplicates(
    ["nome_pessoa", "tipo_pessoa"]
)

dim_people = surrogate_key(
    people,
    ["nome_pessoa", "tipo_pessoa"],
    "sk_person_id"
).select(
    "sk_person_id",
    "nome_pessoa",
    "tipo_pessoa"
)

overwrite_table(dim_people, "dim_people")

display(dim_people.limit(10))


In [0]:

companies = entities.filter(
    F.col("tipo_entidade") == "Produtora"
).select(
    F.col("nome_entidade").alias("nome_produtora")
).filter(
    F.col("nome_produtora").isNotNull()
).dropDuplicates(["nome_produtora"])

dim_companies = surrogate_key(
    companies,
    ["nome_produtora"],
    "sk_company_id"
).select(
    "sk_company_id",
    "nome_produtora"
)

overwrite_table(dim_companies, "dim_companies")

display(dim_companies.limit(10))


In [0]:

reviews_src = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")

reviews = reviews_src.select(
    value_col(
        reviews_src, ["id_filme", "movie_id"],
        "string", True
    ).alias("id_filme"),

    value_col(
        reviews_src, ["nota_usuario", "nota", "rating", "user_rating"],
        "double", True
    ).alias("nota_usuario")
).filter(
    F.col("id_filme").isNotNull()
)

reviews_agg = reviews.groupBy("id_filme").agg(
    F.count(F.lit(1)).cast("int").alias("qtd_avaliacoes_usuarios"),
    F.round(F.avg("nota_usuario"), 2).cast("double").alias(
        "nota_media_usuarios"
    )
)

dim_reviews_base = reviews_agg.join(
    dim_movies.select("id_filme", "sk_movie_id"),
    on="id_filme",
    how="inner"
).select(
    "sk_movie_id",
    "qtd_avaliacoes_usuarios",
    "nota_media_usuarios"
)

dim_reviews = surrogate_key(
    dim_reviews_base,
    ["sk_movie_id"],
    "sk_review_id"
).select(
    "sk_review_id",
    "sk_movie_id",
    "qtd_avaliacoes_usuarios",
    "nota_media_usuarios"
)

overwrite_table(dim_reviews, "dim_reviews")

display(dim_reviews.limit(10))


In [0]:

movie_genres = genres_src.select(
    value_col(
        genres_src, ["id_filme", "movie_id"],
        "string", True
    ).alias("id_filme"),

    F.trim(
        value_col(
            genres_src, ["nome_genero", "genero", "genre", "genres"],
            "string", True
        )
    ).alias("nome_genero")
).filter(
    F.col("id_filme").isNotNull() &
    F.col("nome_genero").isNotNull() &
    (F.col("nome_genero") != "")
).dropDuplicates(["id_filme", "nome_genero"])

bridge_movie_genre = movie_genres.join(
    dim_movies.select("id_filme", "sk_movie_id"),
    on="id_filme",
    how="inner"
).join(
    dim_genres,
    on="nome_genero",
    how="inner"
).select(
    "sk_movie_id",
    "sk_genre_id"
).dropDuplicates()

overwrite_table(bridge_movie_genre, "bridge_movie_genre")

display(bridge_movie_genre.limit(10))


In [0]:

movie_people = entities.filter(
    F.col("tipo_entidade").isin(person_types)
).select(
    "id_filme",
    F.col("nome_entidade").alias("nome_pessoa"),
    F.col("tipo_entidade").alias("tipo_pessoa")
).dropDuplicates(
    ["id_filme", "nome_pessoa", "tipo_pessoa"]
)

bridge_movie_person = movie_people.join(
    dim_movies.select("id_filme", "sk_movie_id"),
    on="id_filme",
    how="inner"
).join(
    dim_people,
    on=["nome_pessoa", "tipo_pessoa"],
    how="inner"
).select(
    "sk_movie_id",
    "sk_person_id"
).dropDuplicates()

overwrite_table(bridge_movie_person, "bridge_movie_person")

display(bridge_movie_person.limit(10))


In [0]:

movie_companies = entities.filter(
    F.col("tipo_entidade") == "Produtora"
).select(
    "id_filme",
    F.col("nome_entidade").alias("nome_produtora")
).dropDuplicates(["id_filme", "nome_produtora"])

bridge_movie_company = movie_companies.join(
    dim_movies.select("id_filme", "sk_movie_id"),
    on="id_filme",
    how="inner"
).join(
    dim_companies,
    on="nome_produtora",
    how="inner"
).select(
    "sk_movie_id",
    "sk_company_id"
).dropDuplicates()

overwrite_table(bridge_movie_company, "bridge_movie_company")

display(bridge_movie_company.limit(10))


In [0]:

financial_src = spark.table(f"{silver_schema}.tb_financeiro_filmes")
metrics_src = spark.table(f"{silver_schema}.tb_metricas_engajamento")


# Consolida as métricas financeiras por filme

financial = financial_src.select(
    value_col(
        financial_src, ["id_filme", "movie_id"],
        "string", True
    ).alias("id_filme"),

    value_col(
        financial_src, ["orcamento_usd", "budget_usd", "budget"],
        "decimal(18,2)"
    ).alias("orcamento_usd"),

    value_col(
        financial_src, ["receita_usd", "revenue_usd", "revenue"],
        "decimal(18,2)"
    ).alias("receita_usd"),

    value_col(
        financial_src, ["lucro_usd", "profit_usd", "profit"],
        "decimal(18,2)"
    ).alias("lucro_usd"),

    value_col(
        financial_src, ["orcamento_brl", "budget_brl"],
        "decimal(18,2)"
    ).alias("orcamento_brl"),

    value_col(
        financial_src, ["receita_brl", "revenue_brl"],
        "decimal(18,2)"
    ).alias("receita_brl"),

    value_col(
        financial_src, ["lucro_brl", "profit_brl"],
        "decimal(18,2)"
    ).alias("lucro_brl")
).filter(
    F.col("id_filme").isNotNull()
).groupBy("id_filme").agg(
    F.max("orcamento_usd").alias("orcamento_usd"),
    F.max("receita_usd").alias("receita_usd"),
    F.max("lucro_usd").alias("lucro_usd"),
    F.max("orcamento_brl").alias("orcamento_brl"),
    F.max("receita_brl").alias("receita_brl"),
    F.max("lucro_brl").alias("lucro_brl")
)


# Consolida as métricas de engajamento por filme

metrics = metrics_src.select(
    value_col(
        metrics_src, ["id_filme", "movie_id"],
        "string", True
    ).alias("id_filme"),

    value_col(
        metrics_src, ["popularidade", "popularity"],
        "double"
    ).alias("popularidade"),

    value_col(
        metrics_src,
        ["nota_media_tmdb", "vote_average_tmdb", "tmdb_rating"],
        "double"
    ).alias("nota_media_tmdb"),

    value_col(
        metrics_src,
        ["qtd_votos_tmdb", "vote_count_tmdb", "tmdb_vote_count"],
        "int"
    ).alias("qtd_votos_tmdb"),

    value_col(
        metrics_src,
        ["nota_media_imdb", "imdb_rating", "rating_imdb"],
        "double"
    ).alias("nota_media_imdb"),

    value_col(
        metrics_src,
        ["qtd_votos_imdb", "imdb_votes", "vote_count_imdb"],
        "int"
    ).alias("qtd_votos_imdb")
).filter(
    F.col("id_filme").isNotNull()
).groupBy("id_filme").agg(
    F.max("popularidade").alias("popularidade"),
    F.max("nota_media_tmdb").alias("nota_media_tmdb"),
    F.max("qtd_votos_tmdb").alias("qtd_votos_tmdb"),
    F.max("nota_media_imdb").alias("nota_media_imdb"),
    F.max("qtd_votos_imdb").alias("qtd_votos_imdb")
)


# Seleciona filmes lançados até a data atual

released_movies = dim_movies.filter(
    F.col("data_lancamento").isNotNull() &
    (F.col("data_lancamento") <= F.current_date())
).select(
    "sk_movie_id",
    "id_filme"
)


# Consolida a fato sem joins com dimensões multivaloradas

fact_movies_performance = released_movies.join(
    financial,
    on="id_filme",
    how="left"
).join(
    metrics,
    on="id_filme",
    how="left"
).select(
    "sk_movie_id",
    F.col("orcamento_usd").cast("decimal(18,2)").alias("orcamento_usd"),
    F.col("receita_usd").cast("decimal(18,2)").alias("receita_usd"),
    F.col("lucro_usd").cast("decimal(18,2)").alias("lucro_usd"),
    F.col("orcamento_brl").cast("decimal(18,2)").alias("orcamento_brl"),
    F.col("receita_brl").cast("decimal(18,2)").alias("receita_brl"),
    F.col("lucro_brl").cast("decimal(18,2)").alias("lucro_brl"),
    F.col("popularidade").cast("double").alias("popularidade"),
    F.col("nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
    F.col("qtd_votos_tmdb").cast("int").alias("qtd_votos_tmdb"),
    F.col("nota_media_imdb").cast("double").alias("nota_media_imdb"),
    F.col("qtd_votos_imdb").cast("int").alias("qtd_votos_imdb")
)

overwrite_table(fact_movies_performance, "fact_movies_performance")

display(fact_movies_performance.limit(10))


In [0]:

tables = [
    "dim_movies",
    "dim_genres",
    "dim_people",
    "dim_companies",
    "dim_reviews",
    "bridge_movie_genre",
    "bridge_movie_person",
    "bridge_movie_company",
    "fact_movies_performance"
]

for table in tables:
    df = spark.table(f"{gold_schema}.{table}")
    print(f"{table}: {df.count()} linhas")


# Verifica o grão da fato

fact = spark.table(f"{gold_schema}.fact_movies_performance")

print("Total de linhas:", fact.count())
print("Filmes distintos:", fact.select("sk_movie_id").distinct().count())

duplicates = (
    fact.groupBy("sk_movie_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Filmes duplicados na fato:", duplicates.count())
display(duplicates.limit(10))


# Verifica unicidade das SKs

for table, sk in [
    ("dim_movies", "sk_movie_id"),
    ("dim_genres", "sk_genre_id"),
    ("dim_people", "sk_person_id"),
    ("dim_companies", "sk_company_id"),
    ("dim_reviews", "sk_review_id")
]:
    df = spark.table(f"{gold_schema}.{table}")

    duplicated_keys = (
        df.groupBy(sk)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    null_keys = df.filter(F.col(sk).isNull()).count()

    print(
        f"{table}: SKs duplicadas={duplicated_keys}, "
        f"SKs nulas={null_keys}"
    )


# Verifica pares duplicados nas tabelas-ponte

for table, keys in [
    ("bridge_movie_genre", ["sk_movie_id", "sk_genre_id"]),
    ("bridge_movie_person", ["sk_movie_id", "sk_person_id"]),
    ("bridge_movie_company", ["sk_movie_id", "sk_company_id"])
]:
    df = spark.table(f"{gold_schema}.{table}")

    duplicates = (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(f"{table}: pares duplicados={duplicates}")


In [0]:

from pyspark.sql import functions as F

# Carrega as tabelas Gold

dim_movies = spark.table(f"{gold_schema}.dim_movies")
dim_genres = spark.table(f"{gold_schema}.dim_genres")
dim_people = spark.table(f"{gold_schema}.dim_people")
dim_companies = spark.table(f"{gold_schema}.dim_companies")
dim_reviews = spark.table(f"{gold_schema}.dim_reviews")

fact = spark.table(f"{gold_schema}.fact_movies_performance")
bridge_genre = spark.table(f"{gold_schema}.bridge_movie_genre")
bridge_person = spark.table(f"{gold_schema}.bridge_movie_person")
bridge_company = spark.table(f"{gold_schema}.bridge_movie_company")


# Integridade referencial

print("=== INTEGRIDADE REFERENCIAL ===")

checks = [
    ("Fato -> Filmes", fact, "sk_movie_id", dim_movies, "sk_movie_id"),
    ("Reviews -> Filmes", dim_reviews, "sk_movie_id", dim_movies, "sk_movie_id"),
    ("Bridge gênero -> Filmes", bridge_genre, "sk_movie_id", dim_movies, "sk_movie_id"),
    ("Bridge gênero -> Gêneros", bridge_genre, "sk_genre_id", dim_genres, "sk_genre_id"),
    ("Bridge pessoa -> Filmes", bridge_person, "sk_movie_id", dim_movies, "sk_movie_id"),
    ("Bridge pessoa -> Pessoas", bridge_person, "sk_person_id", dim_people, "sk_person_id"),
    ("Bridge produtora -> Filmes", bridge_company, "sk_movie_id", dim_movies, "sk_movie_id"),
    ("Bridge produtora -> Produtoras", bridge_company, "sk_company_id", dim_companies, "sk_company_id"),
]

for label, source, source_key, target, target_key in checks:
    orphan_count = (
        source.select(source_key)
        .filter(F.col(source_key).isNotNull())
        .join(
            target.select(target_key).distinct(),
            source[source_key] == target[target_key],
            "left_anti"
        )
        .count()
    )

    null_count = source.filter(F.col(source_key).isNull()).count()

    print(
        f"{label}: registros sem correspondência={orphan_count}, "
        f"FKs nulas={null_count}"
    )


# Cobertura da fato e análise de datas

print("\n=== COBERTURA DA FATO ===")

today = F.current_date()

dim_movies.agg(
    F.count("*").alias("total_dim_movies"),
    F.sum(
        F.when(
            F.col("data_lancamento").isNull(), 1
        ).otherwise(0)
    ).alias("data_nula"),
    F.sum(
        F.when(
            F.col("data_lancamento") > today, 1
        ).otherwise(0)
    ).alias("data_futura"),
    F.sum(
        F.when(
            F.col("data_lancamento").isNotNull() &
            (F.col("data_lancamento") <= today), 1
        ).otherwise(0)
    ).alias("filmes_com_data_valida")
).show(truncate=False)

fact_ids = fact.select("sk_movie_id").distinct()

print("Filmes elegíveis que não entraram na fato:")
dim_movies.filter(
    F.col("data_lancamento").isNotNull() &
    (F.col("data_lancamento") <= today)
).join(
    fact_ids,
    on="sk_movie_id",
    how="left_anti"
).count()

print("Filmes com data nula ou futura, fora da fato:")
dim_movies.filter(
    F.col("data_lancamento").isNull() |
    (F.col("data_lancamento") > today)
).count()


# Métricas financeiras

print("\n=== MÉTRICAS FINANCEIRAS ===")

fact.agg(
    F.count("*").alias("total_filmes"),
    F.sum(F.when(F.col("orcamento_usd").isNull(), 1).otherwise(0)).alias("orcamento_usd_nulo"),
    F.sum(F.when(F.col("receita_usd").isNull(), 1).otherwise(0)).alias("receita_usd_nula"),
    F.sum(F.when(F.col("lucro_usd").isNull(), 1).otherwise(0)).alias("lucro_usd_nulo"),
    F.sum(F.when(F.col("orcamento_brl").isNull(), 1).otherwise(0)).alias("orcamento_brl_nulo"),
    F.sum(F.when(F.col("receita_brl").isNull(), 1).otherwise(0)).alias("receita_brl_nula"),
    F.sum(F.when(F.col("lucro_brl").isNull(), 1).otherwise(0)).alias("lucro_brl_nulo"),
    F.sum(F.when(F.col("orcamento_usd") < 0, 1).otherwise(0)).alias("orcamentos_usd_negativos"),
    F.sum(F.when(F.col("receita_usd") < 0, 1).otherwise(0)).alias("receitas_usd_negativas"),
    F.sum(F.when(F.col("orcamento_brl") < 0, 1).otherwise(0)).alias("orcamentos_brl_negativos"),
    F.sum(F.when(F.col("receita_brl") < 0, 1).otherwise(0)).alias("receitas_brl_negativas"),
    F.sum(
        F.when(
            F.col("lucro_usd").isNotNull() &
            F.col("receita_usd").isNotNull() &
            F.col("orcamento_usd").isNotNull() &
            (F.abs(
                F.col("lucro_usd") -
                (F.col("receita_usd") - F.col("orcamento_usd"))
            ) > 0.02),
            1
        ).otherwise(0)
    ).alias("lucros_usd_inconsistentes"),
    F.sum(
        F.when(
            F.col("lucro_brl").isNotNull() &
            F.col("receita_brl").isNotNull() &
            F.col("orcamento_brl").isNotNull() &
            (F.abs(
                F.col("lucro_brl") -
                (F.col("receita_brl") - F.col("orcamento_brl"))
            ) > 0.02),
            1
        ).otherwise(0)
    ).alias("lucros_brl_inconsistentes")
).show(truncate=False)


# Métricas de engajamento

print("\n=== MÉTRICAS DE ENGAJAMENTO ===")

fact.agg(
    F.sum(
        F.when(
            (F.col("nota_media_tmdb") < 0) |
            (F.col("nota_media_tmdb") > 10), 1
        ).otherwise(0)
    ).alias("notas_tmdb_fora_0_10"),
    F.sum(
        F.when(
            (F.col("nota_media_imdb") < 0) |
            (F.col("nota_media_imdb") > 10), 1
        ).otherwise(0)
    ).alias("notas_imdb_fora_0_10"),
    F.sum(
        F.when(F.col("qtd_votos_tmdb") < 0, 1).otherwise(0)
    ).alias("votos_tmdb_negativos"),
    F.sum(
        F.when(F.col("qtd_votos_imdb") < 0, 1).otherwise(0)
    ).alias("votos_imdb_negativos"),
    F.sum(
        F.when(F.col("popularidade") < 0, 1).otherwise(0)
    ).alias("popularidade_negativa")
).show(truncate=False)


# Integridade e consistência de SKs

print("\n=== CHAVES SUBSTITUTAS ===")

for table_name, key in [
    ("dim_movies", "sk_movie_id"),
    ("dim_genres", "sk_genre_id"),
    ("dim_people", "sk_person_id"),
    ("dim_companies", "sk_company_id"),
    ("dim_reviews", "sk_review_id")
]:
    df = spark.table(f"{gold_schema}.{table_name}")

    result = df.agg(
        F.count("*").alias("linhas"),
        F.countDistinct(key).alias("sk_distintas"),
        F.sum(F.when(F.col(key).isNull(), 1).otherwise(0)).alias("sk_nulas")
    ).first()

    print(
        f"{table_name}: linhas={result['linhas']}, "
        f"SKs distintas={result['sk_distintas']}, "
        f"SKs nulas={result['sk_nulas']}"
    )


# Duplicidade nos pares das tabelas-ponte

print("\n=== DUPLICIDADE NAS PONTES ===")

for table_name, keys in [
    ("bridge_movie_genre", ["sk_movie_id", "sk_genre_id"]),
    ("bridge_movie_person", ["sk_movie_id", "sk_person_id"]),
    ("bridge_movie_company", ["sk_movie_id", "sk_company_id"])
]:
    df = spark.table(f"{gold_schema}.{table_name}")

    duplicates = (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(f"{table_name}: pares duplicados={duplicates}")


# Verifica se as SKs baseadas em hash correspondem às chaves naturais.
# Recalcula o hash a partir das chaves naturais de cada dimensão.

print("\n=== CONSISTÊNCIA DOS HASHES ===")

natural_keys = {
    "dim_movies": (["id_filme"], "sk_movie_id"),
    "dim_genres": (["nome_genero"], "sk_genre_id"),
    "dim_people": (["nome_pessoa", "tipo_pessoa"], "sk_person_id"),
    "dim_companies": (["nome_produtora"], "sk_company_id"),
    "dim_reviews": (["sk_movie_id"], "sk_review_id")
}

for table_name, (keys, sk) in natural_keys.items():
    df = spark.table(f"{gold_schema}.{table_name}")

    hash_key = F.conv(
        F.substring(
            F.sha2(F.to_json(F.struct(*[F.col(c) for c in keys])), 256),
            1,
            15
        ),
        16,
        10
    ).cast("long")

    mismatches = df.filter(
        F.col(sk).isNull() | (F.col(sk) != hash_key)
    ).count()

    print(f"{table_name}: SKs divergentes do hash={mismatches}")
